# AISIN IMRA（米国・imra.com）論文一覧スクレイピング

今井先生の元タスク文書「Build a Researcher Database」の3番目の対象のうち、IMRA America公式サイト。

- 対象URL: `https://www.imra.com/imra-research-papers/`
- 素の`curl`だと406が返るが、ブラウザ相当のヘッダー（Accept/Accept-Language/Referer）を付けると200になる（WAFのuser-agent/ヘッダー判定と思われる）
- robots.txt確認済み（`/wp-admin/`以外は許可）
- 構造: WordPress(Visual Composer)のグリッドレイアウト。1論文＝1つの`div.wpb_row.vc_row`で、内部に`.pub-list`が2つ（左＝タイトル、右＝掲載誌＋DOIリンク）
- **著者名の掲載なし**（このページにはタイトルと掲載誌情報のみで、著者リストがない）。ただしDOIリンクがほぼ全件にあるため、著者情報はOpenAlex/Crossrefへの照会時にDOI経由で別途取得する設計にする

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json

import pandas as pd
import requests
from bs4 import BeautifulSoup

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'AGENTS.md').exists() and (candidate / 'data').is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError('リポジトリ直下を特定できません。リポジトリ内で実行してください。')

OUTPUT_DIR = ROOT / 'data' / 'derived' / 'toyota_official_scrape'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_URL = 'https://www.imra.com/imra-research-papers/'
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': 'https://www.imra.com/',
}

RUN_AT_UTC = datetime.now(timezone.utc).isoformat()
print(f'RUN_AT_UTC = {RUN_AT_UTC}')

RUN_AT_UTC = 2026-07-27T03:03:18.854919+00:00


## 1. 取得

In [2]:
resp = requests.get(SOURCE_URL, headers=HEADERS, timeout=30)
resp.raise_for_status()
print(f'HTTPステータス: {resp.status_code}')
print(f'取得サイズ: {len(resp.text)} 文字')

soup = BeautifulSoup(resp.text, 'html.parser')
rows = soup.select('.wpb_row.vc_row')
print(f'vc_row数: {len(rows)}')

HTTPステータス: 200
取得サイズ: 714171 文字
vc_row数: 350


## 2. 構造化

`.pub-list`が2つある行だけを論文エントリとして扱う（ページ内の見出し・区切り行を除外）。

In [3]:
paper_rows = []
for row in rows:
    pubs = row.select('.pub-list')
    if len(pubs) != 2:
        continue
    title = pubs[0].get_text(strip=True)
    venue_el = pubs[1]
    link_tag = venue_el.select_one('a')
    link = link_tag.get('href') if link_tag else None
    venue_text = venue_el.get_text(strip=True)
    paper_rows.append({'title': title, 'venue': venue_text, 'link': link})

df = pd.DataFrame(paper_rows)
print(f'論文件数: {len(df)}')
print(f'リンクなし: {df["link"].isna().sum()}')
print(f'doi.org以外のリンク: {(~df["link"].fillna("").str.contains("doi.org")).sum()}')
df.head(5)

論文件数: 313
リンクなし: 1
doi.org以外のリンク: 90


,title,venue,link
0,Low noise resonant amplification by optical in...,"Optics Letters, Vol. 51, Issue 13, pp. 3613-36...",https://doi.org/10.1364/OL.603468
1,Long-wave mid-infrared cavity-enhanced frequen...,"Optics Express, Vol. 34, Issue 4, pp. 6751-676...",https://doi.org/10.1364/OE.585687
2,Terahertz amplification by injection locking o...,"IEEE Journal of Quantum Electronics (2026), Vo...",https://doi.org/10.1109/JQE.2026.3652530
3,Dual-wavelength Brillouin lasers as compact op...,"Optics Express, Vol. 33, Issue 20, pp. 41777-4...",https://doi.org/10.1364/OE.569127
4,Fully stabilized Er fiber comb at 1 GHz by har...,"Optica, Vol. 12, Issue 9, pp. 1486-1491 (2025)",https://doi.org/10.1364/OPTICA.568460


## 3. 保存

In [4]:
raw_html_path = OUTPUT_DIR / 'imra_com_raw.html'
raw_html_path.write_text(resp.text, encoding='utf-8')

csv_path = OUTPUT_DIR / 'imra_com.csv'
df.to_csv(csv_path, index=False)

meta_path = OUTPUT_DIR / 'imra_com_meta.json'
with meta_path.open('w', encoding='utf-8') as f:
    json.dump({
        'run_at_utc': RUN_AT_UTC,
        'source_url': SOURCE_URL,
        'n_items': len(df),
        'note': '著者名の記載なし。DOI経由でOpenAlex/Crossrefから別途取得する必要がある。',
    }, f, ensure_ascii=False, indent=2)

print(f'保存: {raw_html_path.relative_to(ROOT)}')
print(f'保存: {csv_path.relative_to(ROOT)}')
print(f'保存: {meta_path.relative_to(ROOT)}')

保存: data/derived/toyota_official_scrape/imra_com_raw.html
保存: data/derived/toyota_official_scrape/imra_com.csv
保存: data/derived/toyota_official_scrape/imra_com_meta.json
